# ML-10 — Content Action Playbook

This notebook turns the validated refresh-scoring work into a practical, human-reviewed action playbook.

Lane: Refresh / Content Opportunity Scoring. The Week-4 baseline prioritizes pages with at least 300 impressions and average position 4–20, scoring them by impressions. This is decision support, not production automation.

## 1. Ranked actions + reason codes

**Primary action:** `review_refresh` — send the highest-scoring pages to an editor for review.

**Reason code:** `visible_position_opportunity` — meaningful search visibility (>=300 impressions) with average position 4–20.

**Fallback:** `monitor` / `insufficient_signal`.

**Archetype → action:** high impressions + position 4–20 → review_refresh; lower impressions or position outside 4–20 → monitor; strong top-3 visibility is not an automatic refresh recommendation.

Freshness evidence is directional/observational: it supports reviewing older content as a possible opportunity, but does not prove that refreshing every old page causes improvement.

In [1]:
!pip -q install duckdb pyarrow
import duckdb, pandas as pd, json, os

# The HF token is already stored in Colab Secrets as HF_TOKEN.
# Load it into DuckDB's Hugging Face secret so hf:// reads are authenticated.
token = os.environ.get('HF_TOKEN')
try:
    from google.colab import userdata
    token = token or userdata.get('HF_TOKEN')
except Exception:
    pass

con = duckdb.connect()
if token:
    safe_token = token.replace("'", "''")
    con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{safe_token}')")
else:
    raise RuntimeError('HF_TOKEN is not available. Add HF_TOKEN to Colab Secrets and enable notebook access.')

rel = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')"
q = f'''SELECT client_hash_id AS client_id, content_hash_id AS content_id,
SUM(gsc_impressions) AS impressions, SUM(gsc_clicks) AS clicks,
AVG(gsc_avg_position) AS avg_position,
BOOL_OR(ga4_data_available IS TRUE) AS ga4_available
FROM {rel} GROUP BY 1,2'''
df = con.execute(q).df()
df['ctr'] = df['clicks'] / df['impressions'].replace(0, pd.NA)
df['qualifies'] = (df['impressions'] >= 300) & df['avg_position'].between(4,20, inclusive='both')
df['score'] = df['impressions'].where(df['qualifies'], 0)
df['reason_code'] = df['qualifies'].map({True:'visible_position_opportunity', False:'insufficient_signal'})
df['action'] = df['qualifies'].map({True:'review_refresh', False:'monitor'})
queue = df.sort_values(['score','impressions'], ascending=False).reset_index(drop=True)
queue.insert(0, 'rank', range(1, len(queue)+1))
os.makedirs('work/outputs', exist_ok=True)
queue.to_csv('work/outputs/baseline_action_score.csv', index=False)
print('Rows:', len(queue))
print('review_refresh:', int((queue.action=='review_refresh').sum()))
print(queue.head(10)[['rank','client_id','content_id','score','avg_position','reason_code','action']].to_string(index=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 331437
review_refresh: 45328
 rank               client_id               content_id    score  avg_position                  reason_code         action
    1 client_23a62021009f63c4 content_e8a52cf3d5988c07 244931.0     15.008339 visible_position_opportunity review_refresh
    2 client_23a62021009f63c4 content_44f34c0a90047651 212404.0      7.346909 visible_position_opportunity review_refresh
    3 client_08a6a72ff48e62c0 content_e7b5dd4dff461ad2 205045.0      4.544203 visible_position_opportunity review_refresh
    4 client_62f4a7e64f5e0096 content_b99ea6861864dea5 194337.0      4.450106 visible_position_opportunity review_refresh
    5 client_73cda7b4e4f265ea content_471d9cabce329a66 164885.0      4.656030 visible_position_opportunity review_refresh
    6 client_73cda7b4e4f265ea content_f43118e089ecc69a 139417.0      5.036458 visible_position_opportunity review_refresh
    7 client_23a62021009f63c4 content_66288edeb93b7c4f 137878.0     18.615742 visible_position_opportunity revi

## 2. Intended use, limits, cost/value

**Intended user:** SEO/content editor.

Use the queue to prioritize human review; it is not an automatic publishing decision.

Limits: the score uses the current measurement window. It does not predict Google's algorithm, guarantee traffic growth, or establish that refresh causes improvement. Search intent, SERP competition, seasonality, technical issues, business value, and editorial quality are not fully represented.

Main cost: editor time. Potential value: better prioritization of limited review capacity. Actual refresh ROI must be measured separately.

In [2]:
print('Top-50 review queue size:', min(50, len(queue)))
print('Top-50 qualifying share:', round((queue.head(50).action=='review_refresh').mean(), 3))
print('Median score among qualifying pages:', queue.loc[queue.qualifies, 'score'].median())
print('Decision rule: impressions >= 300 AND avg_position between 4 and 20')

Top-50 review queue size: 50
Top-50 qualifying share: 1.0
Median score among qualifying pages: 1262.0
Decision rule: impressions >= 300 AND avg_position between 4 and 20


## 3. Human review + no-go list

Before refresh, check search intent, page quality/accuracy, business relevance, current SERP context, cannibalization, technical/indexing issues, and whether the page was recently updated.

**Do NOT automate:** publishing edits; deleting/redirecting pages; canonical/indexing changes; large-scale internal-link changes; causal claims; or refresh solely because the score is high. Keep an explicit human gate.

In [3]:
review = queue.head(10)[['rank','client_id','content_id','score','avg_position','reason_code','action']].copy()
review['human_checks'] = 'intent; quality; business value; SERP context; cannibalization; technical/indexing status'
review['wrong_if'] = 'data is stale, intent changed, technical issue explains performance, or page has already been refreshed'
print(review.to_string(index=False))

 rank               client_id               content_id    score  avg_position                  reason_code         action                                                                              human_checks                                                                                                wrong_if
    1 client_23a62021009f63c4 content_e8a52cf3d5988c07 244931.0     15.008339 visible_position_opportunity review_refresh intent; quality; business value; SERP context; cannibalization; technical/indexing status data is stale, intent changed, technical issue explains performance, or page has already been refreshed
    2 client_23a62021009f63c4 content_44f34c0a90047651 212404.0      7.346909 visible_position_opportunity review_refresh intent; quality; business value; SERP context; cannibalization; technical/indexing status data is stale, intent changed, technical issue explains performance, or page has already been refreshed
    3 client_08a6a72ff48e62c0 content_e7b5dd4dff461a

## 4. Monitoring / retrain triggers

Monitor at least monthly or after material changes in the search environment. Triggers: Precision@K drops materially versus the frozen benchmark; impression/position distributions shift; missingness/availability changes; field definitions or time coverage change; or editors repeatedly reject the same reason code.

Retrain/recalibrate only after a fresh time-aware validation split and evidence that the new version beats the frozen baseline on the same metric and comparable slice. Do not retrain merely because a month passed.

Track queue size, top-K review outcomes, rejection rate, and Precision@K when a valid observed outcome is available.

In [4]:
metrics = {
  'rows_scored': int(len(queue)),
  'review_refresh_rows': int((queue.action=='review_refresh').sum()),
  'top50_review_share': float((queue.head(50).action=='review_refresh').mean()),
  'rule': 'impressions >= 300 AND avg_position between 4 and 20',
  'score': 'impressions for qualifying pages, else 0',
  'decision_status': 'human_review_required'
}
with open('work/outputs/w07_action_playbook_metrics.json','w') as f: json.dump(metrics,f,indent=2)
print(json.dumps(metrics, indent=2))

{
  "rows_scored": 331437,
  "review_refresh_rows": 45328,
  "top50_review_share": 1.0,
  "rule": "impressions >= 300 AND avg_position between 4 and 20",
  "score": "impressions for qualifying pages, else 0",
  "decision_status": "human_review_required"
}


## 5. Exports

The queue CSV is intentionally regenerated by the notebook and is not committed. The metrics JSON is the traceable receipt. Figures, if added, belong under `work/figures/`.

The paper should preserve the distinction between observed associations and causal claims about refreshing content.

In [5]:
print('Exported:', 'work/outputs/baseline_action_score.csv')
print('Exported:', 'work/outputs/w07_action_playbook_metrics.json')
assert os.path.exists('work/outputs/baseline_action_score.csv')
assert os.path.exists('work/outputs/w07_action_playbook_metrics.json')

Exported: work/outputs/baseline_action_score.csv
Exported: work/outputs/w07_action_playbook_metrics.json


## Self-check

- [x] Ranked actions and reason codes are explicit.
- [x] Archetype → action mapping is explicit.
- [x] Decay/refresh insight is directional/observational.
- [x] Intended use, limits, cost/value are stated.
- [x] Human review rules and no-go automation cases are stated.
- [x] Monitoring and retrain triggers are defined.
- [x] Queue and metrics are exported to `work/outputs/`.
- [ ] Run all in Colab and save the executed notebook back to GitHub.

Final wording: practical, non-production content prioritization playbook for human review.